# §67 — Window-geometry read: L5dfiDoTA / L0dfiDoTArr / L5dficapDoTA on 12-row DoTA-CAP-dev windows (s2024)

Rule: `core/docs/final/KIM_RUN.md` **§67**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **Inference only, nothing trained.** In-domain, PRELIMINARY (n = 1); decides only which explanation §64.2's failure gets. K5f stays the reference.

Each row scores DoTA-CAP-dev twice with its own checkpoint: **whole** clip (= protocol B, gated against its saved read-out) and **windowed** (12 rows, hop 6, each window alone, as in training). G = L5 windowed − L5 whole and R = L5 windowed − L0 windowed decide: GEOMETRY / PART GEOMETRY / NOT GEOMETRY.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, preflight | no | 3 min |
| 4 | tests — **raises if red** | no | 1 min |
| 5 | stage DoTA s1 + definitions | no | 3-5 min |
| 6 | read 3 rows × 2 geometries -> **paste `REPORTS/window_geometry/window_geometry_read.md` to Claude** | yes | 15-30 min |

## 1. Mount, paths, preflight

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE   = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_F = '/content/drive/MyDrive/Thesis-Final'  # code, the rows; this notebook's report
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert '## 67. Window-geometry read' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §67 -- re-upload kat-vad/'
assert (REPO / 'core/tools/window_geometry_read.py').is_file(), 'no window_geometry_read -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]
from core import constants  # noqa: E402
from core.data.v2_splits import load_split  # noqa: E402

assert constants.REPO_ROOT == REPO and hasattr(constants, 'KIM_WG_ROWS'), 're-upload kat-vad/'
SEED    = constants.KIM_SEED
K5      = constants.KIM_MAIN_ARM
CAP_DEV = constants.V2_SPLIT_DOTA_CAP_DEV
NAMES   = constants.KIM_CLIP_HOST_ROWS[constants.KIM_CLIP_HOST_DOTA]
HOST    = constants.OUTPUT_ROOT / constants.KIM_CLIP_HOST_OUTPUT_DIRNAME
ROWS    = {'L5': HOST / NAMES['L5'], 'L5cap': HOST / NAMES['L5cap'], 'L0': HOST / (NAMES['L0'] + constants.KIM_ABLATION_RERUN_SUFFIX)}
OUT     = HOST / 'REPORTS' / 'window_geometry'
DOTA    = constants.DATA_ROOT / 'DoTA'
DOTA_S1 = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
LOCAL   = Path('/content/kim_window_geometry')
LOCAL.mkdir(parents=True, exist_ok=True)

need = [DOTA_S1, DOTA / 'metadata_val.json', DOTA / 'val_split.txt', DOTA / 'labels_s8']
for root in ROWS.values():   # L0 = L0dfiDoTArr: §64.2's control
    need += [root / K5 / f's{SEED}' / 'checkpoint_last.pt', root / K5 / f's{SEED}' / 'metrics.jsonl',
             root / K5 / f's{SEED}' / 'config.yaml', root / 'eval' / 'dota' / K5 / 'protocol_b_readout.json']
missing = [str(p) for p in need if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
CAP_IDS = sorted(load_split(CAP_DEV))
miss = [v for v in CAP_IDS if not (DOTA_S1 / f'{v}.npy').exists()]
assert not miss and len(CAP_IDS) == 569, f'{len(miss)} DoTA-CAP-dev clips lack s1 features'
print('rows:', {k: v.name for k, v in ROWS.items()}, '| report ->', OUT)

## 2. Dependencies

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- the read needs one'

## 3. Helpers

In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}')


def stage(src, dst, ids=None):
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    return dst


def publish(src, dst):
    for f in [p for p in src.rglob('*') if p.is_file()]:
        target = dst / f.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        tmp = target.with_name(target.name + '.partial')
        shutil.copy2(f, tmp)
        with open(tmp, 'rb+') as fh:
            os.fsync(fh.fileno())
        os.replace(tmp, target)

## 4. Tests (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_window_geometry_read.py',
     'core/tests/test_rate_matched_eval.py', 'core/tests/test_protocol_b_eval.py'])

## 5. Stage

In [ ]:
t0 = time.time()
L_S1  = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
print(f'staged in {time.time() - t0:.0f}s')

## 6. Read (§67 WG3–WG7) -> **paste `window_geometry_read.md` to Claude**

In [ ]:
import torch  # noqa: E402

assert torch.cuda.is_available(), 'torch sees no GPU'
local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
t0 = time.time()
run([sys.executable, '-m', 'core.tools.window_geometry_read',
     *[x for k, root in ROWS.items() for x in ('--row', k, root.name, root)],
     '--s1-dir', L_S1, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
     '--data-dir', L_LAB, '--seed', SEED, '--device', 'cuda', '--out-dir', local])
publish(local, OUT)
print(f'read in {(time.time() - t0) / 60:.0f} min')
print((OUT / f'{constants.KIM_WG_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()
print('Drive flushed -- paste the report above to Claude')